<a href="https://colab.research.google.com/github/S-ShehryarBurney/YOLOv8-Waste-Detection/blob/main/End_to_End_Waste_Garbage_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIRI Team PITB AI Internship Task 1 — Waste Detection

## Phase 1 — Problem Selection
- **Project Objective**: Implement and evaluate an object detection pipeline for automated waste material sorting and inspection.
- **Application Context**: Controlled material-identification and sorting inspection environments. This project is specifically designed for indoor/sorting facilities and is **not** intended for outdoor street litter, CCTV surveillance, or moving conveyor belts.
- **Approved Classes**: 5 supervisor-approved classes (defined on 18 Nov 2026):
  - `0`: cardboard
  - `1`: glass
  - `2`: metal
  - `3`: paper
  - `4`: plastic
- **Exclusions**: The class `BIODEGRADABLE` was explicitly excluded because it accounted for a vast majority of valid source boxes (45,395 of 74,073) and mostly contained fresh produce, which did not align with the target material-identification goals.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Phase 6 — Setup and Environment

- **Google Drive Storage**: Mounted manually inside Google Colab first to establish persistence, then managed using the programmatic `drive.mount("/content/drive")` code block above to allow seamless, automated runs.
- **Hardware Verification**: Running on a **Tesla T4 GPU** (verified via `nvidia-smi`) to enable accelerated training of deep neural networks.
- **Framework**: Installed and configured the `ultralytics` package (version `8.4.216`) with PyTorch backend supporting GPU execution.

In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [ ]:
import torch

print("GPU Available:", torch.cuda.is_available())
!nvidia-smi

GPU Available: True
Sun Nov 22 19:26:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [ ]:
import ultralytics
from ultralytics import YOLO

print("Ultralytics Version:", ultralytics.__version__)
print("YOLO import successful")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics Version: 8.4.216
YOLO import successful


In [ ]:
!cat /content/drive/MyDrive/cv_project/dataset/data.yaml

path: /content/drive/MyDrive/cv_project/dataset
train: images/train
val: images/val
test: images/test

names:
  0: cardboard
  1: glass
  2: metal
  3: paper
  4: plastic


## Phases 2–5 — Dataset & Annotation
- **Dataset Source**: 300 images sourced from Kaggle/Roboflow Universe (`GARBAGE CLASSIFICATION 3` v2, licensed under CC BY 4.0).
- **Properties**: 416x416 RGB source images, resized at training time to `imgsz=640`.
- **Data Split**: Split in a 70% / 20% / 10% ratio:
  - **Train**: 210 images (210 labels)
  - **Val**: 60 images (60 labels)
  - **Test**: 30 images (30 labels)
- **Annotations**: Contains a total of 1,043 valid YOLO-formatted bounding boxes distributed as follows:
  - `cardboard`: 218 boxes
  - `glass`: 204 boxes
  - `metal`: 245 boxes
  - `paper`: 169 boxes
  - `plastic`: 207 boxes
- **Roboflow Annotation Practice**: Per supervisor instructions and to gain hands-on annotation practice, a separate 10-image manual annotation exercise was successfully completed in Roboflow on 19 Nov 2026. This exercise is kept completely separate from the main 300-image dataset to preserve evaluation integrity.
- **Dataset Configuration**: The `data.yaml` config path points to `/content/drive/MyDrive/cv_project/dataset` containing train, val, and test splits with classes 0 to 4.

In [ ]:
from pathlib import Path

dataset = Path("/content/drive/MyDrive/cv_project/dataset")

for split in ["train", "val", "test"]:
  images = list((dataset / "images" / split).glob("*"))
  labels = list((dataset / "labels" / split).glob("*.txt"))

  print(f"{split.upper()}: Images = {len(images)}, Labels = {len(labels)}")

TRAIN: Images = 210, Labels = 210
VAL: Images = 60, Labels = 60
TEST: Images = 30, Labels = 30


In [ ]:
from collections import Counter

class_counts = Counter()
invalid = []

for split in ["train", "val", "test"]:
  images_dir = dataset / "images" / split
  labels_dir = dataset / "labels" / split

  images = {p.stem for p in images_dir.glob("*.jpg")}
  labels = {p.stem for p in labels_dir.glob("*.txt")}

  if images != labels:
    invalid.append(f"{split}: image-label filename mismatch")

  for file in labels_dir.glob("*.txt"):
    for line_no, line in enumerate(file.read_text().splitlines(), 1):
      values = line.split()

      try:
        if len(values) != 5:
          raise ValueError("Expected 5 values")

        class_id = int(values[0])
        coords = list(map(float, values[1:]))

        if class_id not in range(5):
          raise ValueError("Invalid class ID")

        if not all(0 <= x <= 1 for x in coords):
          raise ValueError("Coordinates not between 0-1")

        if coords[2] <= 0 or coords[3] <= 0:
          raise ValueError("Invalid box dimension")

        class_counts[class_id] += 1

      except ValueError as e:
        invalid.append(f"{file.name}, line {line_no}: {e}")

print("Class Distributions:", class_counts)
print("Invalid annotations:", len(invalid))
print(invalid[:10])

Class Distributions: Counter({2: 245, 0: 218, 4: 207, 1: 204, 3: 169})
Invalid annotations: 0
[]


### YOLOv8n Baseline Training

In [ ]:
from ultralytics import YOLO

n_model = YOLO("yolov8n.pt")

results = n_model.train(
    data = "/content/drive/MyDrive/cv_project/dataset/data.yaml",
    epochs = 30,
    imgsz = 640,
    batch = 8,
    pretrained = True,
    project = "/content/drive/MyDrive/cv_project/outputs/training_results",
    name = "yolov8n_baseline"
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/cv_project/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_baseline, nbs=64, nms=None, op

### YOLOv8s Baseline Training

In [ ]:
from ultralytics import YOLO

s_model = YOLO("yolov8s.pt")

s_results = s_model.train(
    data = "/content/drive/MyDrive/cv_project/dataset/data.yaml",
    epochs = 30,
    imgsz = 640,
    batch = 8,
    pretrained = True,
    project = "/content/drive/MyDrive/cv_project/outputs/training_results",
    name = "yolov8s_baseline"
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/cv_project/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_baseline, nbs=64, nms=None, op

### Model Evaluation on Test Dataset

In [ ]:
from ultralytics import YOLO

n_best_model = YOLO("/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline/weights/best.pt")

n_test_metrics = n_best_model.val(
    data = "/content/drive/MyDrive/cv_project/dataset/data.yaml",
    split = "test",
    imgsz = 640,
    batch = 8,
    project = "/content/drive/MyDrive/cv_project/outputs/training_results",
    name = "yolov8n_test_evaluation"
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,679 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.3 ms, read: 0.0±0.0 MB/s, size: 15.5 KB)
val: Scanning /content/drive/MyDrive/cv_project/dataset/labels/test.cache... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 7.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.3s/it 9.4s
                   all         30         94      0.349      0.422       0.35      0.224
             cardboard          7         36      0.444      0.389      0.286      0.141
                 glass          7         10      0.223        0.4      0.466      0.342
                 metal          6         12      0.442      0.667      0.429      0.338
                 paper          7         22      0.394      0.227       0.27      0.148
               plastic          8  

In [ ]:
from ultralytics import YOLO

s_best_model = YOLO("/content/drive/MyDrive/cv_project/outputs/training_results/yolov8s_baseline/weights/best.pt")

s_test_metrics = s_best_model.val(
    data = "/content/drive/MyDrive/cv_project/dataset/data.yaml",
    split = "test",
    imgsz = 640,
    batch = 8,
    project = "/content/drive/MyDrive/cv_project/outputs/training_results",
    name = "yolov8s_test_evaluation"
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.1 ms, read: 19.0±7.1 MB/s, size: 17.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/cv_project/dataset/labels/test.cache... 30 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30/30 7.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.7it/s 1.5s
                   all         30         94      0.689      0.297      0.397       0.28
             cardboard          7         36       0.41     0.0833      0.219      0.106
                 glass          7         10      0.831        0.5      0.512      0.395
                 metal          6         12      0.827

### Batch Inference on Test Set

In [ ]:
from ultralytics import YOLO

n_predict_model = YOLO("/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline/weights/best.pt")

n_prediction_results = n_predict_model.predict(
    source = "/content/drive/MyDrive/cv_project/dataset/images/test",
    conf = 0.35,
    save = True,
    show_labels = True,
    show_conf = True,
    project = "/content/drive/MyDrive/cv_project/outputs/predictions",
    name = "yolov8n_test_predictions"
)


image 1/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1222_jpg.rf.70022a8317dda0013e0b5e73f006272b.jpg: 640x640 (no detections), 7.6ms
image 2/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1400_jpg.rf.958fc2b5f76e37c7749873ac2ddef85d.jpg: 640x640 1 cardboard, 14.2ms
image 3/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1526_jpg.rf.b73eccbff5b4a14c2bc3774035c0b351.jpg: 640x640 2 cardboards, 7.9ms
image 4/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1668_jpg.rf.2d7b82470787f2143fee2332b54b45c9.jpg: 640x640 3 cardboards, 7.5ms
image 5/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard292_jpg.rf.a3421bb01820e80866f18cf2e516150e.jpg: 640x640 1 cardboard, 7.7ms
image 6/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard644_jpg.rf.06037753390640fa517e787a516020dd.jpg: 640x640 7 cardboards, 8.7ms
image 7/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard876_jpg.rf.de

In [ ]:
from ultralytics import YOLO

s_predict_model = YOLO("/content/drive/MyDrive/cv_project/outputs/training_results/yolov8s_baseline/weights/best.pt")

s_prediction_results = s_predict_model.predict(
    source = "/content/drive/MyDrive/cv_project/dataset/images/test",
    conf = 0.35,
    save = True,
    show_labels = True,
    show_conf = True,
    project = "/content/drive/MyDrive/cv_project/outputs/predictions",
    name = "yolov8s_test_predictions"
)


image 1/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1222_jpg.rf.70022a8317dda0013e0b5e73f006272b.jpg: 640x640 1 cardboard, 1 metal, 16.5ms
image 2/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1400_jpg.rf.958fc2b5f76e37c7749873ac2ddef85d.jpg: 640x640 1 cardboard, 21.9ms
image 3/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1526_jpg.rf.b73eccbff5b4a14c2bc3774035c0b351.jpg: 640x640 1 cardboard, 16.1ms
image 4/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard1668_jpg.rf.2d7b82470787f2143fee2332b54b45c9.jpg: 640x640 2 cardboards, 16.0ms
image 5/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard292_jpg.rf.a3421bb01820e80866f18cf2e516150e.jpg: 640x640 (no detections), 16.0ms
image 6/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard644_jpg.rf.06037753390640fa517e787a516020dd.jpg: 640x640 2 cardboards, 16.1ms
image 7/30 /content/drive/MyDrive/cv_project/dataset/images/test/cardboard

## Phase 7 — Training

To establish a baseline comparison, I trained both models under identical conditions (30 epochs, image size 640, batch size 8, and pre-trained weights) using my custom dataset of 210 training images.

- **YOLOv8n Baseline Training**:
  - **Model**: `yolov8n.pt` (3.2M parameters)
  - **Execution**: Completed successfully on the Tesla T4 GPU.
  - **Logs**: Saved to `/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline`.

- **YOLOv8s Baseline Training**:
  - **Model**: `yolov8s.pt` (11.2M parameters)
  - **Execution**: Completed successfully on the Tesla T4 GPU.
  - **Logs**: Saved to `/content/drive/MyDrive/cv_project/outputs/training_results/yolov8s_baseline`.

## Phase 8 — Evaluation

I evaluated both trained models on my unseen test dataset of 30 images containing 94 annotated target waste items.

### Metrics Guide
- **Precision (P)**: Out of all items the model predicted, how many were actually correct?
- **Recall (R)**: Out of all real items in the image, how many did the model find?
- **mAP50**: Mean Average Precision calculated at an intersection threshold of 0.5.
- **mAP50-95**: The main strict metric calculated across multiple intersection thresholds (from 0.5 to 0.95).

### Test Performance Summary

#### 1. YOLOv8n Test Performance (30 images, 94 instances):
- **Overall Metrics**: Precision = 0.349 | Recall = 0.422 | mAP50 = 0.350 | mAP50-95 = 0.224
- **Class Breakdown**:
  - `cardboard`: P = 0.444 | R = 0.389 | mAP50 = 0.286 | mAP50-95 = 0.141
  - `glass`: P = 0.223 | R = 0.400 | mAP50 = 0.466 | mAP50-95 = 0.342
  - `metal`: P = 0.442 | R = 0.667 | mAP50 = 0.429 | mAP50-95 = 0.338
  - `paper`: P = 0.394 | R = 0.227 | mAP50 = 0.270 | mAP50-95 = 0.148
  - `plastic`: P = 0.239 | R = 0.429 | mAP50 = 0.302 | mAP50-95 = 0.15

#### 2. YOLOv8s Test Performance (30 images, 94 instances):
- **Overall Metrics**: Precision = 0.689 | Recall = 0.297 | mAP50 = 0.397 | mAP50-95 = 0.280
- **Class Breakdown**:
  - `cardboard`: P = 0.410 | R = 0.083 | mAP50 = 0.219 | mAP50-95 = 0.106
  - `glass`: P = 0.831 | R = 0.500 | mAP50 = 0.512 | mAP50-95 = 0.395
  - `metal`: P = 0.827 | R = 0.500 | mAP50 = 0.612 | mAP50-95 = 0.512
  - `paper`: P = 0.599 | R = 0.045 | mAP50 = 0.165 | mAP50-95 = 0.074
  - `plastic`: P = 0.779 | R = 0.357 | mAP50 = 0.478 | mAP50-95 = 0.313

### Analytical Comparison
While YOLOv8n has higher overall recall (0.422 vs 0.297), YOLOv8s has a much higher precision score (0.689 vs 0.349) and reaches better overall box quality with a score of 0.280 compared to YOLOv8n's 0.224.

## Phase 9 — Inference

I ran batch prediction on the 30 unseen test images using both trained checkpoints. To keep the predictions clean, I set a confidence threshold of 0.35.

- **Parameters**: `conf=0.35`, saving boxes, class labels, and confidence levels.
- **YOLOv8n Output**: `/content/drive/MyDrive/cv_project/outputs/predictions/yolov8n_test_predictions`
- **YOLOv8s Output**: `/content/drive/MyDrive/cv_project/outputs/predictions/yolov8s_test_predictions`

I packed both sets of predicted images into zip files (`yolov8n_test_predictions.zip` and `yolov8s_test_predictions.zip`) and downloaded them to my local machine.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/yolov8n_test_predictions",
    "zip",
    "/content/drive/MyDrive/cv_project/outputs/predictions/yolov8n_test_predictions"
)

files.download("/content/yolov8n_test_predictions.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/yolov8s_test_predictions",
    "zip",
    "/content/drive/MyDrive/cv_project/outputs/predictions/yolov8s_test_predictions"
)

files.download("/content/yolov8s_test_predictions.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Phase 10 — Error Analysis and Action Plan

### Detailed Test Set Error Analysis
I reviewed 10 incorrect or weak test predictions (meeting the assignment's minimum of at least 10 reviewed examples) and structured my findings in the table below:

| Image | Actual Object | Model Prediction | Error Type | Possible Reason |
| :--- | :--- | :--- | :--- | :--- |
| cardboard1222 | 3 cardboard | No detections | 3 false negatives | Cardboard items not detected at conf=0.35; confidence may be below threshold. |
| cardboard644 | 28 cardboard | 7 cardboard detections | Many false negatives / under-detection | Dense group of many similar, closely arranged cardboard pieces; overlapping objects make localization difficult. |
| cardboard1668 | 1 cardboard, 6 paper | 3 cardboard detections | Paper missed; possible wrong-class/extra detections | Mixed materials in a cluttered scene may be difficult to separate consistently. |
| glass1685 | 3 glass | No detections | 3 false negatives | Transparent glass and reflections may reduce useful visual features; predictions may fall below threshold. |
| glass303 | 1 glass, 1 metal | 1 metal detection | Glass false negative | Glass bottle missed while metal object was detected. |
| metal1256 | 1 metal | 1 plastic detection | Wrong class | Metal can appearance may have been confused with plastic. |
| metal294 | 1 metal | 1 paper detection | Wrong class | Reflective/crumpled metal surface may resemble paper-like texture. |
| metal78 | 4 metal | 1 metal, 1 glass | Missed objects and wrong class | Closely grouped cans; one object classified as glass and multiple objects missed. |
| paper1660 | 3 paper | No detections | 3 false negatives | Crumpled paper shapes and high-contrast background may make localization difficult. |
| plastic280 | 1 plastic | 1 paper detection | Wrong class | Transparent plastic container may share visual features with light-colored/transparent materials. |

### Analytical Summary of Error Categories
I evaluated my model's behavior against standard error categories including false positives, false negatives, wrong class predictions, poor bounding boxes, small-object detection problems, poor lighting problems, and annotation mistakes.
- **Key Observations**: I observed substantial false negatives, wrong-class predictions, and severe under-detection in crowded images.
- **False Positive Caveat**: I noted that cases showing overlapping or multiple bounding boxes around a single item require careful box-level ground-truth matching before they can be officially counted as false positives.
- **Unconfirmed Error Types**: I did not establish confirmed poor-lighting issues or clear annotation mistakes from this specific prediction set alone.

### My Strategic Recommendations and Action Plan
To address these limitations, I propose the following three concrete improvements:
1. **Diverse Data Expansion**: I will add more diverse training examples for difficult cases (including transparent glass, reflective metal, crumpled paper, plastic containers, and crowded/multi-object scenes) while carefully keeping my validation and test splits entirely separate to preserve evaluation integrity.
2. **Targeted Label Auditing**: I will audit and correct only confirmed annotation issues (such as missing labelled objects, incorrect class IDs, or inaccurate bounding boxes) while preserving the original dataset and recording any newly corrected version separately.
3. **Controlled Model Comparison**: I will run a controlled YOLOv8s comparison on the same dataset and thoroughly compare validation metrics and per-class results against my YOLOv8n baseline.


In [ ]:
import shutil
from pathlib import Path

source_weights = Path("/content/drive/MyDrive/cv_project/outputs/training_results/yolov8s_baseline/weights/best.pt")
dest_dir = Path("/content/drive/MyDrive/cv_project/models")

dest_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(source_weights, dest_dir / "best.pt")
print("Successfully saved best.pt to cv_project/models/best.pt!")

Successfully saved best.pt to cv_project/models/best.pt!


In [ ]:
from pathlib import Path

readme_path = Path("/content/drive/MyDrive/cv_project/README.md")
readme_path.parent.mkdir(parents=True, exist_ok=True)

readme_content = """# Waste Detection System Using YOLOv8 (AIRI Team PITB - AI Internship Task 1)

## Overview
This project implements and evaluates an object detection pipeline for controlled waste material identification and sorting inspection across five supervisor-approved material categories (`cardboard`, `glass`, `metal`, `paper`, and `plastic`). The `BIODEGRADABLE` category was excluded from the project scope.

This pipeline is intended for controlled material-identification and sorting inspection scenarios, and is not designed for outdoor street litter detection, CCTV surveillance, or moving conveyor belts.

## Target Classes
- `0`: `cardboard`
- `1`: `glass`
- `2`: `metal`
- `3`: `paper`
- `4`: `plastic`

## Dataset & Annotation
- **Source**: 300 images from `GARBAGE CLASSIFICATION 3` v2 (Kaggle / Roboflow Universe, licensed under CC BY 4.0), with 416x416 RGB source images resized to `imgsz=640` during training.
- **Split**: 70% / 20% / 10% (`210` train, `60` validation, `30` test) with `1,043` valid YOLO bounding boxes (`metal`: 245, `cardboard`: 218, `plastic`: 207, `glass`: 204, `paper`: 169).
- **Annotation Practice**: Per supervisor clarification allowing a pre-annotated dataset and recommending at least 10 manually annotated images for practice, a separate 10-image manual annotation exercise was completed in Roboflow on 19 Nov 2026 and kept separate from the 300-image dataset.

## Training & Evaluation Summary
Both models were trained for 30 epochs with `imgsz=640`, `batch=8`, and `pretrained=True` on an NVIDIA Tesla T4 GPU using `ultralytics` (`8.4.216`).

### Validation Metrics (60 images, 323 instances)
- **YOLOv8n (`yolov8n_baseline`)**: Precision = `0.311` | Recall = `0.238` | mAP@0.5 = `0.207` | mAP@0.5:0.95 = `0.143`
- **YOLOv8s (`yolov8s_baseline`)**: Precision = `0.358` | Recall = `0.219` | mAP@0.5 = `0.213` | mAP@0.5:0.95 = `0.154`

### Test Metrics (30 images, 94 instances)
- **YOLOv8n (`yolov8n_test_evaluation`)**: Precision = `0.349` | Recall = `0.422` | mAP@0.5 = `0.350` | mAP@0.5:0.95 = `0.224`
- **YOLOv8s (`yolov8s_test_evaluation`)**: Precision = `0.689` | Recall = `0.297` | mAP@0.5 = `0.397` | mAP@0.5:0.95 = `0.280`

## Inference & Error Analysis
- Batch inference was run on all 30 unseen test images at `conf=0.35` with bounding boxes, class labels, and confidence scores saved to `outputs/predictions/yolov8n_test_predictions` and `outputs/predictions/yolov8s_test_predictions`.
- A manual error analysis of 10 weak or incorrect test predictions was completed alongside improvement recommendations covering diverse training examples for difficult/crowded items, targeted label auditing on confirmed issues, and controlled model comparison.

## Project Structure
~~~text
cv_project/
├── dataset/
│   ├── data.yaml
│   ├── images/ (train, val, test)
│   └── labels/ (train, val, test)
├── models/
│   └── best.pt
├── notebooks/
│   └── End_to_End_Waste_Garbage_Detection_System.ipynb
├── outputs/
│   ├── predictions/
│   └── training_results/
├── report/
│   └── final_report.pdf
├── README.md
└── requirements.txt
~~~
"""

readme_path.write_text(readme_content, encoding="utf-8")
print("Updated:", readme_path, "Size:", readme_path.stat().st_size, "bytes")

Updated: /content/drive/MyDrive/cv_project/README.md Size: 3178 bytes


In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/cv_project")
checks = [
    base / "dataset/data.yaml",
    base / "models/best.pt",
    base / "README.md",
    base / "report/final_report.pdf",
    base / "outputs/training_results/yolov8s_baseline/results.png",
    base / "outputs/training_results/yolov8s_baseline/confusion_matrix.png",
]

for p in checks:
    print(f"{'OK' if p.exists() else 'MISSING'}: {p}")

for pred_dir in [
    base / "outputs/predictions/yolov8n_test_predictions",
    base / "outputs/predictions/yolov8s_test_predictions",
]:
    imgs = list(pred_dir.glob("*.jpg")) + list(pred_dir.glob("*.png"))
    print(f"Predictions in {pred_dir.name}: {len(imgs)} images")

OK: /content/drive/MyDrive/cv_project/dataset/data.yaml
OK: /content/drive/MyDrive/cv_project/models/best.pt
OK: /content/drive/MyDrive/cv_project/README.md
MISSING: /content/drive/MyDrive/cv_project/report/final_report.pdf
OK: /content/drive/MyDrive/cv_project/outputs/training_results/yolov8s_baseline/results.png
OK: /content/drive/MyDrive/cv_project/outputs/training_results/yolov8s_baseline/confusion_matrix.png
Predictions in yolov8n_test_predictions: 30 images
Predictions in yolov8s_test_predictions: 30 images
